## Step 9 — block population assignment
**# of cells in notebook:** 6 (!)

**Purpose:** This is a long notebook with lots of code that certainly needs to be refactored with a helper module…at this point that remains a to-do. The ultimate purpose of this notebook is to assign population to blocks. First we identify the best population raster, using basic tests, research, and personal judgement. In Juba, South Sudan that was Landscan Mosaic. We extract raster cells that overlap the Juba extent. We convert to point and rebuild a vector grid. Needless to say, the population grid does not line up with block boundaries. We apportion a grid cell’s population to overlapping block areas  in proportion to the amount of building footprint in the overlapping area, a type of dasymetric allocation. We also download Overture building footprints for grid cells that intersect the urban extent, but have areas outside the urban extent. Doing so ensures that proportional allocation by building footprint area considers all footprints across a grid cell. We sum the allocated populations over block IDs to obtain block populations.           

**Input:**

- a geodatabase with a dissolved blocks layer in WGS84 (create it if it doesn’t already exist)
- a population raster   
- buildings layer from World Bank or GRID3
- overture buildings for areas of grid cells that intersect the extent but are outside the extent. 
- blocks layer

**Output:** 
- all intermediate layers to allow for for QA/QC checks 
- blocks layer with population column

**Main logic:**

**Cell 1 — Create the population grid from the LandScan raster**
1. Uses the existing dissolved block/settlement extent, `blocks_dissolve_wgs84`, as the study-area extent for the population workflow. 
2. Creates an envelope around that extent, clips the LandScan raster to the rectangular extent, and replaces NoData cells with `0`. 
3. Converts the cleaned raster to points so that each raster cell’s population value can be transferred to vector geometry. 
4. Creates a fishnet polygon grid matching the raster cell size and extent, then spatially joins the raster-point values to the fishnet polygons to create `pop_grid`. 
5. Selects the `pop_grid` cells that intersect the settlement/block extent, saves them as `pop_grid_settlement_extent`, and then erases the actual settlement extent from those cells to create `pop_grid_settlement_erase`. 

**Cell 2 — Reproject population-grid layers to UTM**
1. Takes the three population-grid layers from Cell 1: `pop_grid`, `pop_grid_settlement_extent`, and `pop_grid_settlement_erase`. 
2. Projects each of those layers to WGS 1984 UTM Zone 36N (Juba). 
3. Deletes the original versions of those layers and renames the projected versions back to the original layer names. 
4. The result is that the same three layer names remain in the geodatabase, but they are now in UTM Zone 36N. 

**Cell 3 — Download Overture buildings for the erased outer grid area**
1. Reads `pop_grid_settlement_erase`, cleans invalid or empty geometries, and reprojects it to WGS84 for use as an Overture download area. 
2. Builds a bounding box around the erased population-grid area and uses the Overture Maps CLI to download building features within that bounding box. 
3. Clips the downloaded Overture buildings to the exact `pop_grid_settlement_erase` polygons. 
4. Saves the clipped Overture buildings to a GeoPackage, rereads them, reprojects them to UTM, and writes them back to the population geodatabase as `overture_pop_grid_settlement_erase`. 

**Cell 4 — Intersect buildings, blocks, and population-grid cells**
1. Removes Overture building features whose `names` field contains “airport,” then selects the erased population-grid cells that intersect the remaining Overture buildings. 
2. Intersects those selected outer grid cells with the Overture buildings to create `overture_outside`, and calculates building area in square meters for those outside-settlement building pieces. 
3. Clips `pop_grid` to the dissolved settlement/block extent to create `pop_grid_extent_clip`. 
4. Intersects the World Bank building polygons with the clipped population grid to create `buildings_inside`. 
5. Intersects the block polygons with the full pop_grid to create the block/grid crosswalk layer `itx_blocks_pop_grid`. 
6. Intersects `buildings_inside` with `itx_blocks_pop_grid` to create `itx_buildings_inside_blocks_pop_grid`, then calculates `area_m2_utm` for each building/grid/block intersection piece. 

**Cell 5 — Allocate population from grid cells to blocks using building-area shares**
1. Combines inside-building area and outside/Overture-building area by population grid cell to create `pop_grid_building_area_summary`, which stores each grid cell’s population and total building area. 
2. Creates `pop_grid_block_table`, a unique lookup table of which population grid cells intersect which blocks, using the original block `OBJECTID` carried through the intersect. 
3. Adds `building_area_m2` to each grid-cell/block pair by summing the World Bank building area inside that specific grid-cell/block combination. 
4. Adds the total building area for each full population grid cell, then calculates each block’s share of the grid cell’s building area as:
`bldg_area_share = building_area_m2 / grid_total_bldg_area_m2`
5. Multiplies each grid-cell/block share by the grid-cell population to calculate `allocated_pop`. 
6. Sums `allocated_pop` by block to create the final `block_population` table, and exports several QA/output CSVs. 

**Cell 6 — Join allocated population back to the blocks layer**
1. Reads the `block_population` table created in Cell 5. 
2. Builds a lookup from block `OBJECTID` to `SUM_allocated_pop`. 
3. Adds a `population` field to the original blocks layer if it does not already exist. 
4. Updates each block’s `population` field using the matched value from `block_population`. 
5. Blocks with no matching population record are assigned `population = 0`.


In [ ]:
import arcpy
import os
from arcpy.sa import Con, IsNull

# -------------------------------------------------------------------
# Environment
# -------------------------------------------------------------------
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

# Check out Spatial Analyst
arcpy.CheckOutExtension("Spatial")

# -------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------

# Output geodatabase for intermediate and final outputs
gdb = r"E:\World Bank deliverbale 1\_analysis\population\population.gdb"

# IMPORTANT:
# This is already the dissolved settlement/block extent in WGS84.
extent_fc = r"E:\World Bank deliverbale 1\_analysis\roads\roads.gdb\blocks_dissolve_wgs84"

# Outputs
envelope_fc = os.path.join(gdb, "envelope")
raster_clip = os.path.join(gdb, "raster_clip")
raster_clip_zero = os.path.join(gdb, "raster_clip_zero")
raster_point = os.path.join(gdb, "raster_point")
fishnet_temp = os.path.join(gdb, "fishnet_temp")
pop_grid = os.path.join(gdb, "pop_grid")
pop_grid_settlement_extent = os.path.join(gdb, "pop_grid_settlement_extent")
pop_grid_settlement_erase = os.path.join(gdb, "pop_grid_settlement_erase")

# LandScan / WorldPop / popuation raster path
in_raster = r"E:\World Bank deliverbale 1\rasters\population\landscan-mosaic-southsudan-v1-assets\landscan-mosaic-southsudan-v1.tif"

# -------------------------------------------------------------------
# Helper functions
# -------------------------------------------------------------------
def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)
        print(f"Deleted existing: {path}")

def require_exists(path, label):
    if not arcpy.Exists(path):
        raise ValueError(f"{label} not found: {path}")

def print_spatial_reference(path, label):
    desc = arcpy.Describe(path)
    sr = desc.spatialReference
    print(f"{label} spatial reference:")
    print(f"  Name: {sr.name}")
    print(f"  Factory code: {sr.factoryCode}")

# -------------------------------------------------------------------
# Check required inputs
# -------------------------------------------------------------------
require_exists(extent_fc, "Input dissolved extent feature class")
require_exists(in_raster, "Input raster")

print("\nInput checks passed.")
print(f"Using dissolved extent directly: {extent_fc}")
print(f"Using raster: {in_raster}")

print_spatial_reference(extent_fc, "Extent")
print_spatial_reference(in_raster, "Raster")

# -------------------------------------------------------------------
# Cleanup of outputs
# -------------------------------------------------------------------
# IMPORTANT:
# Do NOT include extent_fc here because it is now an input/source layer.
for fc in [
    envelope_fc,
    raster_clip,
    raster_clip_zero,
    raster_point,
    fishnet_temp,
    pop_grid,
    pop_grid_settlement_extent,
    pop_grid_settlement_erase
]:
    delete_if_exists(fc)

# -------------------------------------------------------------------
# Step 1. Use existing dissolved extent
# -------------------------------------------------------------------
print("\nStep 1: Skipping dissolve.")
print("Using existing dissolved WGS84 extent as extent_fc:")
print(f"  {extent_fc}")

# -------------------------------------------------------------------
# Step 2. Create minimum bounding geometry (envelope)
# -------------------------------------------------------------------
print("\nStep 2: Creating Minimum Bounding Geometry (ENVELOPE)...")

arcpy.management.MinimumBoundingGeometry(
    in_features=extent_fc,
    out_feature_class=envelope_fc,
    geometry_type="ENVELOPE",
    group_option="ALL"
)

print(f"Envelope created: {envelope_fc}")

# -------------------------------------------------------------------
# Step 3. Clip raster using extent
# -------------------------------------------------------------------
print("\nStep 3: Clipping raster to rectangular extent of extent feature class...")

arcpy.management.Clip(
    in_raster=in_raster,
    rectangle="#",
    out_raster=raster_clip,
    in_template_dataset=extent_fc,
    nodata_value="-200",
    clipping_geometry="NONE",
    maintain_clipping_extent="NO_MAINTAIN_EXTENT"
)

print(f"Raster clip created: {raster_clip}")

# -------------------------------------------------------------------
# Step 4. Replace NoData with 0
# -------------------------------------------------------------------
print("\nStep 4: Replacing NoData cells with 0...")

raster_zero_obj = Con(IsNull(raster_clip), 0, raster_clip)
raster_zero_obj.save(raster_clip_zero)

print(f"Raster with NoData replaced by 0 created: {raster_clip_zero}")

# -------------------------------------------------------------------
# Step 5. Convert cleaned raster to points
# -------------------------------------------------------------------
print("\nStep 5: Converting raster to points...")

arcpy.conversion.RasterToPoint(
    in_raster=raster_clip_zero,
    out_point_features=raster_point,
    raster_field="Value"
)

print(f"Raster successfully converted to points: {raster_point}")

# -------------------------------------------------------------------
# Step 6. Create fishnet and spatial join raster values to polygons
# -------------------------------------------------------------------
print("\nStep 6: Reading raster properties...")

desc = arcpy.Describe(raster_clip_zero)
extent = desc.extent
spatial_ref = desc.spatialReference
cell_width = desc.meanCellWidth
cell_height = desc.meanCellHeight

print("Raster clip properties:")
print(f"  Extent XMin: {extent.XMin}")
print(f"  Extent YMin: {extent.YMin}")
print(f"  Extent XMax: {extent.XMax}")
print(f"  Extent YMax: {extent.YMax}")
print(f"  Cell width:  {cell_width}")
print(f"  Cell height: {cell_height}")
print(f"  Spatial reference: {spatial_ref.name}")

origin_coord = f"{extent.XMin} {extent.YMin}"
y_axis_coord = f"{extent.XMin} {extent.YMin + 1}"
corner_coord = f"{extent.XMax} {extent.YMax}"

print("\nCreating fishnet...")

arcpy.management.CreateFishnet(
    out_feature_class=fishnet_temp,
    origin_coord=origin_coord,
    y_axis_coord=y_axis_coord,
    cell_width=cell_width,
    cell_height=cell_height,
    number_rows="",
    number_columns="",
    corner_coord=corner_coord,
    labels="NO_LABELS",
    template=raster_clip_zero,
    geometry_type="POLYGON"
)

arcpy.management.DefineProjection(fishnet_temp, spatial_ref)

print(f"Fishnet created: {fishnet_temp}")

print("\nSpatial joining raster values to fishnet...")

arcpy.analysis.SpatialJoin(
    target_features=fishnet_temp,
    join_features=raster_point,
    out_feature_class=pop_grid,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT"
)

count_grid = int(arcpy.management.GetCount(pop_grid)[0])
count_points = int(arcpy.management.GetCount(raster_point)[0])

print(f"pop_grid feature count: {count_grid}")
print(f"raster_point feature count: {count_points}")

fields = [f.name for f in arcpy.ListFields(pop_grid)]
candidate_fields = [f for f in fields if f.upper() in ["GRID_CODE", "GRIDCODE", "VALUE"]]

if candidate_fields:
    value_field = candidate_fields[0]
    null_count = 0
    zero_count = 0

    with arcpy.da.SearchCursor(pop_grid, [value_field]) as cursor:
        for row in cursor:
            if row[0] is None:
                null_count += 1
            elif row[0] == 0:
                zero_count += 1

    print(f"Joined value field: {value_field}")
    print(f"Cells with null joined value: {null_count}")
    print(f"Cells with zero value: {zero_count}")
else:
    print("Could not automatically identify the joined raster value field.")
    print("Please inspect the pop_grid attribute table.")
    print("Available fields:")
    for f in fields:
        print(f"  {f}")

# -------------------------------------------------------------------
# Step 7. Select pop_grid cells intersecting extent
# -------------------------------------------------------------------
print("\nStep 7: Selecting pop_grid cells that intersect settlement extent...")

pop_grid_lyr = "pop_grid_lyr"

if arcpy.Exists(pop_grid_lyr):
    arcpy.management.Delete(pop_grid_lyr)

arcpy.management.MakeFeatureLayer(pop_grid, pop_grid_lyr)

arcpy.management.SelectLayerByLocation(
    in_layer=pop_grid_lyr,
    overlap_type="INTERSECT",
    select_features=extent_fc,
    selection_type="NEW_SELECTION"
)

selected_count = int(arcpy.management.GetCount(pop_grid_lyr)[0])
print(f"Selected features: {selected_count}")

arcpy.conversion.ExportFeatures(
    in_features=pop_grid_lyr,
    out_features=pop_grid_settlement_extent
)

print(f"Exported selected cells: {pop_grid_settlement_extent}")

# -------------------------------------------------------------------
# Step 8. Erase extent from selected pop_grid cells
# -------------------------------------------------------------------
print("\nStep 8: Running Pairwise Erase...")

arcpy.analysis.PairwiseErase(
    in_features=pop_grid_settlement_extent,
    erase_features=extent_fc,
    out_feature_class=pop_grid_settlement_erase
)

print(f"Erase output created: {pop_grid_settlement_erase}")

# -------------------------------------------------------------------
# Final message
# -------------------------------------------------------------------
print("\nWorkflow complete.")
print("Outputs created:")
print(f"  input extent: {extent_fc}")
print(f"  envelope: {envelope_fc}")
print(f"  raster_clip: {raster_clip}")
print(f"  raster_clip_zero: {raster_clip_zero}")
print(f"  raster_point: {raster_point}")
print(f"  fishnet_temp: {fishnet_temp}")
print(f"  pop_grid: {pop_grid}")
print(f"  pop_grid_settlement_extent: {pop_grid_settlement_extent}")
print(f"  pop_grid_settlement_erase: {pop_grid_settlement_erase}")

# -------------------------------------------------------------------
# Check in Spatial Analyst
# -------------------------------------------------------------------
arcpy.CheckInExtension("Spatial")

In [ ]:
import arcpy
import os

# -------------------------------------------------------------------
# Environment
# -------------------------------------------------------------------
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

# -------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------
gdb = r"E:\World Bank deliverbale 1\_analysis\population\population.gdb"

pop_grid = os.path.join(gdb, "pop_grid")
pop_grid_settlement_extent = os.path.join(gdb, "pop_grid_settlement_extent")
pop_grid_settlement_erase = os.path.join(gdb, "pop_grid_settlement_erase")

# Temporary projected outputs
pop_grid_utm36n_tmp = os.path.join(gdb, "pop_grid_utm36n_tmp")
pop_grid_settlement_extent_utm36n_tmp = os.path.join(gdb, "pop_grid_settlement_extent_utm36n_tmp")
pop_grid_settlement_erase_utm36n_tmp = os.path.join(gdb, "pop_grid_settlement_erase_utm36n_tmp")

# -------------------------------------------------------------------
# Helper functions
# -------------------------------------------------------------------
def require_exists(path, label):
    if not arcpy.Exists(path):
        raise ValueError(f"{label} not found: {path}")

def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)
        print(f"Deleted existing: {path}")

def print_crs(path, label):
    desc = arcpy.Describe(path)
    sr = desc.spatialReference
    print(f"{label}:")
    print(f"  Path: {path}")
    print(f"  CRS name: {sr.name}")
    print(f"  Factory code: {sr.factoryCode}")

# -------------------------------------------------------------------
# Check inputs
# -------------------------------------------------------------------
require_exists(pop_grid, "pop_grid")
require_exists(pop_grid_settlement_extent, "pop_grid_settlement_extent")
require_exists(pop_grid_settlement_erase, "pop_grid_settlement_erase")

print("\nInput CRS before projection:")
for fc in [
    pop_grid,
    pop_grid_settlement_extent,
    pop_grid_settlement_erase
]:
    print_crs(fc, os.path.basename(fc))

# -------------------------------------------------------------------
# UTM Zone 36N spatial reference
# -------------------------------------------------------------------
utm36n_sr = arcpy.SpatialReference(32636)

print("\nTarget CRS:")
print(f"  Name: {utm36n_sr.name}")
print(f"  Factory code: {utm36n_sr.factoryCode}")

# -------------------------------------------------------------------
# Delete temp outputs if they exist
# -------------------------------------------------------------------
for fc in [
    pop_grid_utm36n_tmp,
    pop_grid_settlement_extent_utm36n_tmp,
    pop_grid_settlement_erase_utm36n_tmp
]:
    delete_if_exists(fc)

# -------------------------------------------------------------------
# Project to UTM Zone 36N
# -------------------------------------------------------------------
print("\nProjecting pop_grid to UTM Zone 36N...")

arcpy.management.Project(
    in_dataset=pop_grid,
    out_dataset=pop_grid_utm36n_tmp,
    out_coor_system=utm36n_sr
)

print("Projecting pop_grid_settlement_extent to UTM Zone 36N...")

arcpy.management.Project(
    in_dataset=pop_grid_settlement_extent,
    out_dataset=pop_grid_settlement_extent_utm36n_tmp,
    out_coor_system=utm36n_sr
)

print("Projecting pop_grid_settlement_erase to UTM Zone 36N...")

arcpy.management.Project(
    in_dataset=pop_grid_settlement_erase,
    out_dataset=pop_grid_settlement_erase_utm36n_tmp,
    out_coor_system=utm36n_sr
)

# -------------------------------------------------------------------
# Replace original feature classes with UTM Zone 36N versions
# -------------------------------------------------------------------
print("\nReplacing original layers with UTM Zone 36N versions...")

delete_if_exists(pop_grid)
delete_if_exists(pop_grid_settlement_extent)
delete_if_exists(pop_grid_settlement_erase)

arcpy.management.Rename(
    in_data=pop_grid_utm36n_tmp,
    out_data=os.path.basename(pop_grid)
)

arcpy.management.Rename(
    in_data=pop_grid_settlement_extent_utm36n_tmp,
    out_data=os.path.basename(pop_grid_settlement_extent)
)

arcpy.management.Rename(
    in_data=pop_grid_settlement_erase_utm36n_tmp,
    out_data=os.path.basename(pop_grid_settlement_erase)
)

# -------------------------------------------------------------------
# Final check
# -------------------------------------------------------------------
print("\nDone. Updated layers now have the original names:")
print(pop_grid)
print(pop_grid_settlement_extent)
print(pop_grid_settlement_erase)

print("\nOutput CRS after projection:")
for fc in [
    pop_grid,
    pop_grid_settlement_extent,
    pop_grid_settlement_erase
]:
    print_crs(fc, os.path.basename(fc))

print("\nWorkflow complete.")

In [ ]:
import os
import subprocess
import geopandas as gpd

# -----------------------------
# INPUTS
# -----------------------------
gdb_path = r"E:\World Bank deliverbale 1\_analysis\population\population.gdb"
erase_layer = "pop_grid_settlement_erase"

out_dir = r"E:\World Bank deliverbale 1\_analysis\populationc\overture_download"
os.makedirs(out_dir, exist_ok=True)

erase_gpkg = os.path.join(out_dir, "pop_grid_settlement_erase.gpkg")
overture_raw = os.path.join(out_dir, "overture_pop_grid_settlement_erase_raw.geoparquet")
overture_clipped_gpkg = os.path.join(out_dir, "overture_pop_grid_settlement_erase.gpkg")

final_layer_name = "overture_pop_grid_settlement_erase"
out_layer = "overture_pop_grid_settlement_erase"

# -----------------------------
# 1. READ ERASE FEATURES FROM FILE GDB
# -----------------------------
erase_gdf = gpd.read_file(gdb_path, layer=erase_layer)

print("Input CRS:", erase_gdf.crs)
print("Feature count:", len(erase_gdf))

# -----------------------------
# 2. KEEP ONLY VALID, NON-EMPTY GEOMETRIES
# -----------------------------
erase_gdf = erase_gdf[erase_gdf.geometry.notna()].copy()
erase_gdf = erase_gdf[~erase_gdf.geometry.is_empty].copy()
erase_gdf = erase_gdf[erase_gdf.is_valid].copy()

# -----------------------------
# 3. REPROJECT TO WGS84 FOR OVERTURE BBOX DOWNLOAD
# -----------------------------
if erase_gdf.crs is None:
    raise ValueError("Input layer has no CRS defined. Define it before continuing.")

erase_wgs84 = erase_gdf.to_crs(4326)

# Save AOI polygons to GeoPackage
erase_wgs84.to_file(erase_gpkg, layer=erase_layer, driver="GPKG")
print("Saved AOI polygons to:", erase_gpkg)

# -----------------------------
# 4. BUILD BBOX FOR OVERTURE DOWNLOAD
# -----------------------------
minx, miny, maxx, maxy = erase_wgs84.total_bounds
bbox_str = f"{minx},{miny},{maxx},{maxy}"
print("BBox:", bbox_str)

# -----------------------------
# 5. DOWNLOAD OVERTURE BUILDINGS
# -----------------------------
cmd = [
    "overturemaps",
    "download",
    "--bbox", bbox_str,
    "-f", "geoparquet",
    "--type", "building",
    "-o", overture_raw
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("Downloaded raw Overture buildings to:")
print(overture_raw)

# -----------------------------
# 6. READ RAW OVERTURE BUILDINGS
# -----------------------------
bldg_gdf = gpd.read_parquet(overture_raw)

if bldg_gdf.crs is None:
    bldg_gdf = bldg_gdf.set_crs(4326)

print("Raw buildings:", len(bldg_gdf))

# Optional CRS alignment to silence the harmless warning
bldg_gdf = bldg_gdf.to_crs(erase_wgs84.crs)

# -----------------------------
# 7. CLIP BUILDINGS TO THE EXACT ERASE POLYGONS
# -----------------------------
bldg_clip = gpd.clip(bldg_gdf, erase_wgs84)

print("Clipped buildings:", len(bldg_clip))

# -----------------------------
# 8. SAVE FINAL OUTPUT TO GPKG
# -----------------------------
bldg_clip.to_file(overture_clipped_gpkg, layer=final_layer_name, driver="GPKG")

print("Final output saved to:")
print(overture_clipped_gpkg)
print("Layer name:", final_layer_name)

# -----------------------------
# 9. RE-READ THE GPKG LAYER
#    This preserves the successful behavior of your original workflow.
# -----------------------------
gdf = gpd.read_file(overture_clipped_gpkg, layer=final_layer_name)

print("Re-read GPKG CRS:", gdf.crs)
print("Re-read GPKG feature count:", len(gdf))

# -----------------------------
# 10. PROJECT TO WORLD MOLLWEIDE
# -----------------------------
gdf_utm = gdf.to_crs("EPSG:32636")

print("Output CRS:", gdf_moll.crs)

# -----------------------------
# 11. WRITE TO FILE GDB
# -----------------------------
gdf_moll.to_file(gdb_path, layer=out_layer, driver="OpenFileGDB")

print("Written to:")
print(f"{gdb_path}\\{out_layer}")

In [ ]:
import arcpy
import os

# -------------------------------------------------------------------
# Environment
# -------------------------------------------------------------------
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

# -------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------
gdb = r"E:\World Bank deliverbale 1\_analysis\population\population.gdb"

# Existing population-grid inputs from earlier steps
overture_pop_grid_settlement_erase = os.path.join(gdb, "overture_pop_grid_settlement_erase")
pop_grid_settlement_erase = os.path.join(gdb, "pop_grid_settlement_erase")
pop_grid_settlement_extent = os.path.join(gdb, "pop_grid_settlement_extent")
pop_grid = os.path.join(gdb, "pop_grid")

# Existing dissolved settlement/block extent.
# This replaces the old os.path.join(gdb, "extent") input.
extent_fc = r"E:\World Bank deliverbale 1\_analysis\roads\roads.gdb\blocks_dissolve_wgs84"

# UTM 36N source layers
buildings = r"E:\World Bank deliverbale 1\wb_buildings\buildings_area.gpkg\main.buildings"
blocks = r"E:\World Bank deliverbale 1\_analysis\blocks\blocks.gdb\juba_blocks_20260415_small_utm36n"

# Outputs / intermediates
outer_pop_grid = os.path.join(gdb, "outer_pop_grid")
overture_outside = os.path.join(gdb, "overture_outside")
pop_grid_extent_clip = os.path.join(gdb, "pop_grid_extent_clip")
buildings_inside = os.path.join(gdb, "buildings_inside")
itx_blocks_pop_grid = os.path.join(gdb, "itx_blocks_pop_grid")
itx_buildings_inside_blocks_pop_grid = os.path.join(gdb, "itx_buildings_inside_blocks_pop_grid")

# -------------------------------------------------------------------
# Helper functions
# -------------------------------------------------------------------
def require_exists(path, label):
    if not arcpy.Exists(path):
        raise ValueError(f"{label} not found: {path}")

def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)
        print(f"Deleted existing: {path}")

def field_names(fc):
    return [f.name for f in arcpy.ListFields(fc)]

def require_field(fc, field_name, label):
    fields = field_names(fc)
    if field_name not in fields:
        print(f"\nFields in {label}:")
        for f in fields:
            print(f"  {f}")
        raise ValueError(f"Required field '{field_name}' not found in {label}: {fc}")

def add_field_if_missing(fc, field_name, field_type="DOUBLE"):
    existing_fields = field_names(fc)
    if field_name not in existing_fields:
        print(f"Adding field: {field_name}")
        arcpy.management.AddField(fc, field_name, field_type)
    else:
        print(f"Field already exists: {field_name}")

def print_crs(path, label):
    desc = arcpy.Describe(path)
    sr = desc.spatialReference
    print(f"{label} CRS:")
    print(f"  Name: {sr.name}")
    print(f"  Factory code: {sr.factoryCode}")

def print_count(path, label):
    count = int(arcpy.management.GetCount(path)[0])
    print(f"{label} count: {count}")

def print_field_sample(fc, label, max_fields=40):
    fields = field_names(fc)
    print(f"\nFields in {label} ({len(fields)} total):")
    for f in fields[:max_fields]:
        print(f"  {f}")
    if len(fields) > max_fields:
        print(f"  ... {len(fields) - max_fields} more fields not shown")

# -------------------------------------------------------------------
# Spatial reference
# -------------------------------------------------------------------
utm_36n = arcpy.SpatialReference(32636)

# -------------------------------------------------------------------
# Check required inputs
# -------------------------------------------------------------------
require_exists(overture_pop_grid_settlement_erase, "Overture erase layer")
require_exists(pop_grid_settlement_erase, "pop_grid_settlement_erase")
require_exists(pop_grid_settlement_extent, "pop_grid_settlement_extent")
require_exists(pop_grid, "pop_grid")
require_exists(extent_fc, "extent_fc / blocks_dissolve_wgs84")
require_exists(buildings, "buildings")
require_exists(blocks, "blocks")

# Field checks that are most likely to matter
require_field(overture_pop_grid_settlement_erase, "names", "overture_pop_grid_settlement_erase")
require_field(buildings, "area_m_utm", "buildings")

print("\nInput checks passed.")

print("\nInput CRS checks:")
print_crs(overture_pop_grid_settlement_erase, "overture_pop_grid_settlement_erase")
print_crs(pop_grid_settlement_erase, "pop_grid_settlement_erase")
print_crs(pop_grid_settlement_extent, "pop_grid_settlement_extent")
print_crs(pop_grid, "pop_grid")
print_crs(extent_fc, "extent_fc / blocks_dissolve_wgs84")
print_crs(buildings, "buildings")
print_crs(blocks, "blocks")

print("\nInput feature counts:")
print_count(overture_pop_grid_settlement_erase, "overture_pop_grid_settlement_erase")
print_count(pop_grid_settlement_erase, "pop_grid_settlement_erase")
print_count(pop_grid_settlement_extent, "pop_grid_settlement_extent")
print_count(pop_grid, "pop_grid")
print_count(extent_fc, "extent_fc / blocks_dissolve_wgs84")
print_count(buildings, "buildings")
print_count(blocks, "blocks")

print_field_sample(overture_pop_grid_settlement_erase, "overture_pop_grid_settlement_erase")
print_field_sample(pop_grid, "pop_grid")
print_field_sample(extent_fc, "extent_fc / blocks_dissolve_wgs84")
print_field_sample(buildings, "buildings")
print_field_sample(blocks, "blocks")

# -------------------------------------------------------------------
# Optional cleanup of outputs
# -------------------------------------------------------------------
for fc in [
    outer_pop_grid,
    overture_outside,
    pop_grid_extent_clip,
    buildings_inside,
    itx_blocks_pop_grid,
    itx_buildings_inside_blocks_pop_grid
]:
    delete_if_exists(fc)

# -------------------------------------------------------------------
# Step 1. Delete overture features where names contains 'airport'
# -------------------------------------------------------------------
print("\nStep 1: Deleting features containing 'airport' from overture_pop_grid_settlement_erase...")

layer = "temp_overture_layer"

if arcpy.Exists(layer):
    arcpy.management.Delete(layer)

arcpy.management.MakeFeatureLayer(overture_pop_grid_settlement_erase, layer)

where_clause = "UPPER(names) LIKE '%AIRPORT%'"

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=layer,
    selection_type="NEW_SELECTION",
    where_clause=where_clause
)

count = int(arcpy.management.GetCount(layer)[0])
print(f"Features selected for deletion: {count}")

if count > 0:
    print("Deleting selected features...")
    arcpy.management.DeleteFeatures(layer)
else:
    print("No features to delete.")

print("Overture dataset updated in place.")

# -------------------------------------------------------------------
# Step 2. Select pop_grid_settlement_erase cells intersecting overture layer
# -------------------------------------------------------------------
print("\nStep 2: Selecting pop_grid_settlement_erase features that intersect overture layer...")

layer = "pop_grid_erase_lyr"

if arcpy.Exists(layer):
    arcpy.management.Delete(layer)

arcpy.management.MakeFeatureLayer(pop_grid_settlement_erase, layer)

arcpy.management.SelectLayerByLocation(
    in_layer=layer,
    overlap_type="INTERSECT",
    select_features=overture_pop_grid_settlement_erase,
    selection_type="NEW_SELECTION"
)

selected_count = int(arcpy.management.GetCount(layer)[0])
print(f"Selected features: {selected_count}")

arcpy.conversion.ExportFeatures(
    in_features=layer,
    out_features=outer_pop_grid
)

print(f"Output created: {outer_pop_grid}")
print_count(outer_pop_grid, "outer_pop_grid")

# -------------------------------------------------------------------
# Step 3. Intersect outer_pop_grid with overture layer
# -------------------------------------------------------------------
print("\nStep 3: Running Pairwise Intersect for outer_pop_grid and overture layer...")

arcpy.analysis.PairwiseIntersect(
    in_features=[outer_pop_grid, overture_pop_grid_settlement_erase],
    out_feature_class=overture_outside,
    join_attributes="ALL",
    output_type="INPUT"
)

print(f"Output created: {overture_outside}")
print_count(overture_outside, "overture_outside")

# -------------------------------------------------------------------
# Step 4. Add area field and calculate area on overture_outside
# -------------------------------------------------------------------
print("\nStep 4: Calculating area_m2_utm for overture_outside...")

add_field_if_missing(overture_outside, "area_m2_utm", "DOUBLE")

arcpy.management.CalculateGeometryAttributes(
    in_features=overture_outside,
    geometry_property=[["area_m2_utm", "AREA"]],
    area_unit="SQUARE_METERS",
    coordinate_system=utm_36n
)

print("Done.")

# -------------------------------------------------------------------
# Step 5. Clip pop_grid by dissolved settlement/block extent
# -------------------------------------------------------------------
print("\nStep 5: Clipping pop_grid by dissolved settlement/block extent...")

arcpy.analysis.Clip(
    in_features=pop_grid,
    clip_features=extent_fc,
    out_feature_class=pop_grid_extent_clip
)

print(f"Output created: {pop_grid_extent_clip}")
print_count(pop_grid_extent_clip, "pop_grid_extent_clip")

# -------------------------------------------------------------------
# Step 6. Intersect UTM buildings with pop_grid_extent_clip
# -------------------------------------------------------------------
print("\nStep 6: Intersecting UTM buildings with pop_grid_extent_clip...")

arcpy.analysis.PairwiseIntersect(
    in_features=[buildings, pop_grid_extent_clip],
    out_feature_class=buildings_inside,
    join_attributes="ALL",
    output_type="INPUT"
)

print(f"Output created: {buildings_inside}")
print_count(buildings_inside, "buildings_inside")
print_field_sample(buildings_inside, "buildings_inside")

# -------------------------------------------------------------------
# Step 7. Intersect UTM blocks with pop_grid
# -------------------------------------------------------------------
print("\nStep 7: Intersecting UTM blocks with pop_grid...")

arcpy.analysis.PairwiseIntersect(
    in_features=[blocks, pop_grid],
    out_feature_class=itx_blocks_pop_grid,
    join_attributes="ALL",
    output_type="INPUT"
)

print(f"Output created: {itx_blocks_pop_grid}")
print_count(itx_blocks_pop_grid, "itx_blocks_pop_grid")
print_field_sample(itx_blocks_pop_grid, "itx_blocks_pop_grid")

# -------------------------------------------------------------------
# Step 8. Intersect buildings_inside with itx_blocks_pop_grid
# -------------------------------------------------------------------
print("\nStep 8: Intersecting buildings_inside with itx_blocks_pop_grid...")

arcpy.analysis.PairwiseIntersect(
    in_features=[buildings_inside, itx_blocks_pop_grid],
    out_feature_class=itx_buildings_inside_blocks_pop_grid,
    join_attributes="ALL",
    output_type="INPUT"
)

print(f"Output created: {itx_buildings_inside_blocks_pop_grid}")
print_count(itx_buildings_inside_blocks_pop_grid, "itx_buildings_inside_blocks_pop_grid")
print_field_sample(itx_buildings_inside_blocks_pop_grid, "itx_buildings_inside_blocks_pop_grid")

# -------------------------------------------------------------------
# Step 9. Add area field and calculate area on final intersect output
# -------------------------------------------------------------------
print("\nStep 9: Calculating area_m2_utm for final intersect output...")

add_field_if_missing(itx_buildings_inside_blocks_pop_grid, "area_m2_utm", "DOUBLE")

arcpy.management.CalculateGeometryAttributes(
    in_features=itx_buildings_inside_blocks_pop_grid,
    geometry_property=[["area_m2_utm", "AREA"]],
    area_unit="SQUARE_METERS",
    coordinate_system=utm_36n
)

print("Done.")

# -------------------------------------------------------------------
# Final summary
# -------------------------------------------------------------------
print("\nWorkflow complete.")
print("Outputs created / updated:")
print(f"  overture_pop_grid_settlement_erase updated in place: {overture_pop_grid_settlement_erase}")
print(f"  outer_pop_grid: {outer_pop_grid}")
print(f"  overture_outside: {overture_outside}")
print(f"  pop_grid_extent_clip: {pop_grid_extent_clip}")
print(f"  buildings_inside: {buildings_inside}")
print(f"  itx_blocks_pop_grid: {itx_blocks_pop_grid}")
print(f"  itx_buildings_inside_blocks_pop_grid: {itx_buildings_inside_blocks_pop_grid}")

In [ ]:
import arcpy
import pandas as pd
import os

# -------------------------------------------------------------------
# Environment
# -------------------------------------------------------------------
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

# -------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------
gdb = r"E:\World Bank deliverbale 1\_analysis\population\population.gdb"

# Inputs from previous script
buildings_inside = os.path.join(gdb, "itx_buildings_inside_blocks_pop_grid")
buildings_outside = os.path.join(gdb, "overture_outside")
itx_blocks_pop_grid = os.path.join(gdb, "itx_blocks_pop_grid")

# Intermediate / outputs
pop_grid_building_area_summary = os.path.join(gdb, "pop_grid_building_area_summary")
pop_grid_block_table = os.path.join(gdb, "pop_grid_block_table")
tmp_pop_grid_block_bldg_area_sum = os.path.join(gdb, "tmp_pop_grid_block_bldg_area_sum")
pop_grid_block_table_with_bldg_area = os.path.join(gdb, "pop_grid_block_table_with_bldg_area")
block_population = os.path.join(gdb, "block_population")

# CSV outputs
csv_folder = r"E:\World Bank deliverbale 1\population_assignment"

summary_csv = os.path.join(csv_folder, "pop_grid_building_area_summary.csv")
grid_block_csv = os.path.join(csv_folder, "pop_grid_block_table.csv")
block_population_csv = os.path.join(csv_folder, "block_population.csv")

# -------------------------------------------------------------------
# Current field names
# -------------------------------------------------------------------
grid_id_field = "TARGET_FID"
pop_field = "grid_code"
area_field = "area_m2_utm"

# IMPORTANT:
# We are using the original OBJECTID from the input blocks layer.
# After PairwiseIntersect, ArcGIS preserved that source OBJECTID as:
# FID_juba_blocks_20260415_small_utm36n
#
# This replaces the older field:
#   FID_blocks_mollweide
#
# And it also replaces the newer alternative:
#   block_id
block_id_field = "FID_juba_blocks_20260415_small_utm36n"

# -------------------------------------------------------------------
# Helper functions
# -------------------------------------------------------------------
def require_exists(path, label):
    if not arcpy.Exists(path):
        raise ValueError(f"{label} not found: {path}")

def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)
        print(f"Deleted existing: {path}")

def field_names(table):
    return [f.name for f in arcpy.ListFields(table)]

def require_field(table, field_name, label):
    fields = field_names(table)

    if field_name not in fields:
        print(f"\nFields in {label}:")
        for f in fields:
            print(f"  {f}")

        raise ValueError(
            f"Required field '{field_name}' not found in {label}: {table}"
        )

def require_fields(table, fields, label):
    for field in fields:
        require_field(table, field, label)

def add_field_if_missing(table, field_name, field_type):
    existing_fields = field_names(table)

    if field_name not in existing_fields:
        arcpy.management.AddField(
            in_table=table,
            field_name=field_name,
            field_type=field_type
        )
        print(f"Added field: {field_name}")
    else:
        print(f"Field already exists: {field_name}")

def fc_to_df(fc, fields):
    rows = [row for row in arcpy.da.SearchCursor(fc, fields)]
    return pd.DataFrame(rows, columns=fields)

def print_count(path, label):
    count = int(arcpy.management.GetCount(path)[0])
    print(f"{label} count: {count}")

def print_fields(table, label, max_fields=60):
    fields = field_names(table)

    print(f"\nFields in {label} ({len(fields)} total):")
    for f in fields[:max_fields]:
        print(f"  {f}")

    if len(fields) > max_fields:
        print(f"  ... {len(fields) - max_fields} more fields not shown")

# -------------------------------------------------------------------
# Create CSV output folder if needed
# -------------------------------------------------------------------
os.makedirs(csv_folder, exist_ok=True)

# -------------------------------------------------------------------
# Check required inputs
# -------------------------------------------------------------------
require_exists(buildings_inside, "Input feature class buildings_inside")
require_exists(buildings_outside, "Input feature class buildings_outside")
require_exists(itx_blocks_pop_grid, "Input feature class itx_blocks_pop_grid")

# Required fields based on current workflow
require_fields(
    buildings_inside,
    [grid_id_field, pop_field, block_id_field, area_field],
    "buildings_inside / itx_buildings_inside_blocks_pop_grid"
)

require_fields(
    buildings_outside,
    [grid_id_field, pop_field, area_field],
    "buildings_outside / overture_outside"
)

require_fields(
    itx_blocks_pop_grid,
    [grid_id_field, block_id_field],
    "itx_blocks_pop_grid"
)

print("\nInput checks passed.")

print_count(buildings_inside, "buildings_inside / itx_buildings_inside_blocks_pop_grid")
print_count(buildings_outside, "buildings_outside / overture_outside")
print_count(itx_blocks_pop_grid, "itx_blocks_pop_grid")

print_fields(buildings_inside, "buildings_inside / itx_buildings_inside_blocks_pop_grid")
print_fields(buildings_outside, "buildings_outside / overture_outside")
print_fields(itx_blocks_pop_grid, "itx_blocks_pop_grid")

# -------------------------------------------------------------------
# Optional cleanup of outputs
# -------------------------------------------------------------------
for path in [
    pop_grid_building_area_summary,
    pop_grid_block_table,
    tmp_pop_grid_block_bldg_area_sum,
    pop_grid_block_table_with_bldg_area,
    block_population
]:
    delete_if_exists(path)

# -------------------------------------------------------------------
# Step 1. Build pop_grid_building_area_summary
# -------------------------------------------------------------------
print("\nStep 1: Building pop_grid_building_area_summary...")

df_build = fc_to_df(buildings_inside, [grid_id_field, pop_field, area_field])
df_outer = fc_to_df(buildings_outside, [grid_id_field, pop_field, area_field])

df_build = df_build[df_build[grid_id_field].notna()].copy()
df_outer = df_outer[df_outer[grid_id_field].notna()].copy()

df_pop_all = pd.concat(
    [
        df_build[[grid_id_field, pop_field]],
        df_outer[[grid_id_field, pop_field]]
    ],
    ignore_index=True
)

pop_check = (
    df_pop_all[df_pop_all[pop_field].notna()]
    .groupby(grid_id_field)[pop_field]
    .nunique()
    .reset_index(name="n_grid_code")
)

bad = pop_check[pop_check["n_grid_code"] > 1]

if not bad.empty:
    print("ERROR: Some TARGET_FID values have more than one non-null grid_code across the two layers.")
    print(bad.head(20))
    raise ValueError(
        "grid_code is not unique within some TARGET_FID values across the input layers."
    )

pop_df = (
    df_pop_all[df_pop_all[pop_field].notna()]
    .groupby(grid_id_field, as_index=False)[pop_field]
    .first()
    .rename(columns={pop_field: "population"})
)

build_sum = (
    df_build.groupby(grid_id_field, as_index=False)[area_field]
    .sum()
    .rename(columns={area_field: "bldg_area_main_m2"})
)

outer_sum = (
    df_outer.groupby(grid_id_field, as_index=False)[area_field]
    .sum()
    .rename(columns={area_field: "bldg_area_outer_m2"})
)

summary = pd.merge(pop_df, build_sum, on=grid_id_field, how="outer")
summary = pd.merge(summary, outer_sum, on=grid_id_field, how="outer")

for col in ["bldg_area_main_m2", "bldg_area_outer_m2"]:
    summary[col] = summary[col].fillna(0)

summary["total_bldg_area_m2"] = (
    summary["bldg_area_main_m2"] + summary["bldg_area_outer_m2"]
)

summary = summary[
    [
        grid_id_field,
        "population",
        "bldg_area_main_m2",
        "bldg_area_outer_m2",
        "total_bldg_area_m2"
    ]
]

summary = summary.sort_values(grid_id_field).reset_index(drop=True)

print(summary.head())
print(f"Rows in output table: {len(summary)}")

summary.to_csv(summary_csv, index=False)
print(f"CSV written to: {summary_csv}")

arcpy.conversion.TableToTable(
    in_rows=summary_csv,
    out_path=gdb,
    out_name="pop_grid_building_area_summary"
)

print(f"GDB table written to: {pop_grid_building_area_summary}")

# -------------------------------------------------------------------
# Step 2. Build pop_grid_block_table
# -------------------------------------------------------------------
print("\nStep 2: Building pop_grid_block_table...")

rows = [
    row for row in arcpy.da.SearchCursor(
        itx_blocks_pop_grid,
        [grid_id_field, block_id_field]
    )
]

# Standardize output column names.
# block_ID now stores the original OBJECTID from the blocks layer,
# carried through the intersect as FID_juba_blocks_20260415_small_utm36n.
df = pd.DataFrame(rows, columns=["grid_ID", "block_ID"])

df = df[df["grid_ID"].notna()].copy()
df = df[df["block_ID"].notna()].copy()

df_unique = df.drop_duplicates().copy()
df_unique = df_unique.sort_values(["grid_ID", "block_ID"]).reset_index(drop=True)

print(df_unique.head())
print(f"Rows in output table: {len(df_unique)}")

df_unique.to_csv(grid_block_csv, index=False)
print(f"CSV written to: {grid_block_csv}")

arcpy.conversion.TableToTable(
    in_rows=grid_block_csv,
    out_path=gdb,
    out_name="pop_grid_block_table"
)

print(f"GDB table written to: {pop_grid_block_table}")

# -------------------------------------------------------------------
# Step 3. Add building_area_m2 to pop_grid_block_table
# -------------------------------------------------------------------
print("\nStep 3: Building pop_grid_block_table_with_bldg_area...")

arcpy.management.CopyRows(
    pop_grid_block_table,
    pop_grid_block_table_with_bldg_area
)

print(f"Copied input table to output table: {pop_grid_block_table_with_bldg_area}")

arcpy.analysis.Statistics(
    in_table=buildings_inside,
    out_table=tmp_pop_grid_block_bldg_area_sum,
    statistics_fields=[[area_field, "SUM"]],
    case_field=[grid_id_field, block_id_field]
)

print(f"Summary table created: {tmp_pop_grid_block_bldg_area_sum}")

sum_area_field = f"SUM_{area_field}"

require_fields(
    tmp_pop_grid_block_bldg_area_sum,
    [grid_id_field, block_id_field, sum_area_field],
    "tmp_pop_grid_block_bldg_area_sum"
)

pair_to_area = {}

with arcpy.da.SearchCursor(
    tmp_pop_grid_block_bldg_area_sum,
    [grid_id_field, block_id_field, sum_area_field]
) as cursor:
    for grid_id, block_id, area_val in cursor:
        if grid_id is None or block_id is None:
            continue

        if area_val is None:
            area_val = 0

        pair_to_area[(str(grid_id), str(block_id))] = area_val

print(f"Built dictionary for {len(pair_to_area)} unique grid-block pairs.")

add_field_if_missing(
    pop_grid_block_table_with_bldg_area,
    "building_area_m2",
    "DOUBLE"
)

matched = 0
missing = 0

with arcpy.da.UpdateCursor(
    pop_grid_block_table_with_bldg_area,
    ["grid_ID", "block_ID", "building_area_m2"]
) as cursor:
    for row in cursor:
        grid_id = row[0]
        block_id = row[1]

        if grid_id is None or block_id is None:
            row[2] = 0
            missing += 1
        else:
            key = (str(grid_id), str(block_id))

            if key in pair_to_area:
                row[2] = pair_to_area[key]
                matched += 1
            else:
                row[2] = 0
                missing += 1

        cursor.updateRow(row)

print("Done updating building area field.")
print(f"Matched rows: {matched}")
print(f"Rows set to 0: {missing}")

row_count = int(arcpy.management.GetCount(pop_grid_block_table_with_bldg_area)[0])
print(f"Rows in output table: {row_count}")
print(f"Final output table: {pop_grid_block_table_with_bldg_area}")

# -------------------------------------------------------------------
# Step 4. Add grid_total_bldg_area_m2 and flag
# -------------------------------------------------------------------
print("\nStep 4: Writing grid_total_bldg_area_m2 and flag...")

add_field_if_missing(
    pop_grid_block_table_with_bldg_area,
    "grid_total_bldg_area_m2",
    "DOUBLE"
)

add_field_if_missing(
    pop_grid_block_table_with_bldg_area,
    "flag",
    "SHORT"
)

grid_to_area = {}

with arcpy.da.SearchCursor(
    pop_grid_building_area_summary,
    [grid_id_field, "total_bldg_area_m2"]
) as cursor:
    for grid_id, total_area in cursor:
        if grid_id is not None:
            grid_to_area[str(grid_id)] = 0 if total_area is None else total_area

print(f"Lookup dictionary built with {len(grid_to_area)} grid values.")

found_count = 0
missing_count = 0

with arcpy.da.UpdateCursor(
    pop_grid_block_table_with_bldg_area,
    ["grid_ID", "grid_total_bldg_area_m2", "flag"]
) as cursor:
    for row in cursor:
        grid_id = row[0]

        if grid_id is not None and str(grid_id) in grid_to_area:
            row[1] = grid_to_area[str(grid_id)]
            row[2] = 0
            found_count += 1
        else:
            row[1] = 0
            row[2] = 1
            missing_count += 1

        cursor.updateRow(row)

print("Done.")
print(f"Matched grid values: {found_count}")
print(f"Missing grid values: {missing_count}")

# -------------------------------------------------------------------
# Step 5. Calculate bldg_area_share
# -------------------------------------------------------------------
print("\nStep 5: Calculating bldg_area_share...")

add_field_if_missing(
    pop_grid_block_table_with_bldg_area,
    "bldg_area_share",
    "DOUBLE"
)

zero_div_cases = 0

with arcpy.da.UpdateCursor(
    pop_grid_block_table_with_bldg_area,
    ["building_area_m2", "grid_total_bldg_area_m2", "bldg_area_share"]
) as cursor:
    for row in cursor:
        num = row[0]
        den = row[1]

        if num is None:
            num = 0

        if den is None:
            den = 0

        if den == 0:
            row[2] = 0
            zero_div_cases += 1
        else:
            row[2] = num / den

        cursor.updateRow(row)

print("Done.")
print(f"Rows with denominator = 0, set to 0: {zero_div_cases}")

# -------------------------------------------------------------------
# Step 6. Add grid_pop and allocated_pop
# -------------------------------------------------------------------
print("\nStep 6: Writing grid_pop and allocated_pop...")

add_field_if_missing(
    pop_grid_block_table_with_bldg_area,
    "grid_pop",
    "DOUBLE"
)

add_field_if_missing(
    pop_grid_block_table_with_bldg_area,
    "allocated_pop",
    "DOUBLE"
)

grid_to_pop = {}

with arcpy.da.SearchCursor(
    pop_grid_building_area_summary,
    [grid_id_field, "population"]
) as cursor:
    for grid_id, pop_val in cursor:
        if grid_id is not None:
            grid_to_pop[str(grid_id)] = 0 if pop_val is None else pop_val

print(f"Lookup dictionary built with {len(grid_to_pop)} grid values.")

matched_count = 0
missing_count = 0

with arcpy.da.UpdateCursor(
    pop_grid_block_table_with_bldg_area,
    ["grid_ID", "bldg_area_share", "grid_pop", "allocated_pop"]
) as cursor:
    for row in cursor:
        grid_id = row[0]
        share = row[1]

        if share is None:
            share = 0

        if grid_id is not None and str(grid_id) in grid_to_pop:
            grid_pop = grid_to_pop[str(grid_id)]
            matched_count += 1
        else:
            grid_pop = 0
            missing_count += 1

        allocated_pop = share * grid_pop

        row[2] = grid_pop
        row[3] = allocated_pop

        cursor.updateRow(row)

print("Done.")
print(f"Matched grid values: {matched_count}")
print(f"Missing grid values: {missing_count}")

# -------------------------------------------------------------------
# Step 7. Summarize allocated_pop by block_ID
# -------------------------------------------------------------------
print("\nStep 7: Creating block_population...")

arcpy.analysis.Statistics(
    in_table=pop_grid_block_table_with_bldg_area,
    out_table=block_population,
    statistics_fields=[["allocated_pop", "SUM"]],
    case_field=["block_ID"]
)

print(f"Summary table created: {block_population}")

sum_field = "SUM_allocated_pop"
existing_fields = field_names(block_population)

if sum_field in existing_fields:
    arcpy.management.AlterField(
        in_table=block_population,
        field=sum_field,
        new_field_name="sum_allocated_pop",
        new_field_alias="sum_allocated_pop"
    )
    print("Field renamed to sum_allocated_pop")
else:
    print(f"Expected summary field not found: {sum_field}")
    print_fields(block_population, "block_population")

# Export final block population table to CSV
if os.path.exists(block_population_csv):
    os.remove(block_population_csv)

arcpy.conversion.TableToTable(
    in_rows=block_population,
    out_path=os.path.dirname(block_population_csv),
    out_name=os.path.basename(block_population_csv)
)

print(f"CSV written to: {block_population_csv}")

# -------------------------------------------------------------------
# Final summary
# -------------------------------------------------------------------
print("\nWorkflow complete.")
print("Outputs created:")
print(f"  pop_grid_building_area_summary: {pop_grid_building_area_summary}")
print(f"  pop_grid_block_table: {pop_grid_block_table}")
print(f"  pop_grid_block_table_with_bldg_area: {pop_grid_block_table_with_bldg_area}")
print(f"  block_population: {block_population}")
print("CSV outputs:")
print(f"  {summary_csv}")
print(f"  {grid_block_csv}")
print(f"  {block_population_csv}")

In [ ]:
import arcpy
import os

# -------------------------------------------------------------------
# Environment
# -------------------------------------------------------------------
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

# -------------------------------------------------------------------
# Inputs
# -------------------------------------------------------------------
pop_table = r"E:\World Bank deliverbale 1\_analysis\population\population.gdb\block_population"

blocks = r"E:\World Bank deliverbale 1\_analysis\blocks\blocks.gdb\juba_blocks_20260415_small_utm36n"

# -------------------------------------------------------------------
# Field names
# -------------------------------------------------------------------
pop_table_block_id_field = "OBJECTID"
pop_table_value_field = "SUM_allocated_pop"

blocks_join_field = "OBJECTID"
blocks_output_field = "population"

# -------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------
def require_exists(path, label):
    if not arcpy.Exists(path):
        raise ValueError(f"{label} not found: {path}")

def field_names(table):
    return [f.name for f in arcpy.ListFields(table)]

def require_field(table, field_name, label):
    fields = field_names(table)
    if field_name not in fields:
        print(f"\nFields in {label}:")
        for f in fields:
            print(f"  {f}")
        raise ValueError(f"Required field not found: {field_name}")

def add_field_if_missing(table, field_name, field_type="DOUBLE"):
    fields = field_names(table)
    if field_name not in fields:
        arcpy.management.AddField(table, field_name, field_type)
        print(f"Added field: {field_name}")
    else:
        print(f"Field already exists: {field_name}")

# -------------------------------------------------------------------
# Checks
# -------------------------------------------------------------------
require_exists(pop_table, "Population table")
require_exists(blocks, "Blocks layer")

require_field(pop_table, pop_table_block_id_field, "population table")
require_field(pop_table, pop_table_value_field, "population table")
require_field(blocks, blocks_join_field, "blocks layer")

add_field_if_missing(blocks, blocks_output_field, "DOUBLE")

# -------------------------------------------------------------------
# Build lookup dictionary from population table
# -------------------------------------------------------------------
print("\nBuilding population lookup...")

pop_lookup = {}

with arcpy.da.SearchCursor(
    pop_table,
    [pop_table_block_id_field, pop_table_value_field]
) as cursor:
    for block_id, pop_val in cursor:
        if block_id is None:
            continue

        if pop_val is None:
            pop_val = 0

        pop_lookup[int(block_id)] = float(pop_val)

print(f"Population lookup records: {len(pop_lookup)}")

# -------------------------------------------------------------------
# Update blocks population field
# -------------------------------------------------------------------
print("\nUpdating blocks population field...")

matched = 0
missing = 0

with arcpy.da.UpdateCursor(
    blocks,
    [blocks_join_field, blocks_output_field]
) as cursor:
    for oid, population in cursor:
        if oid in pop_lookup:
            population = pop_lookup[oid]
            matched += 1
        else:
            population = 0
            missing += 1

        cursor.updateRow([oid, population])

print("\nDone.")
print(f"Matched blocks: {matched}")
print(f"Blocks with no population match, set to 0: {missing}")
print(f"Updated layer: {blocks}")
print(f"Updated field: {blocks_output_field}")